### Projet de Machine Learning 2026 - Chiara, Augustin, Anatole

In [2]:
import pandas as pd
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split


# 1. Chemins


# À MODIFIER : chemin vers le dossier du projet
DIR = Path("/Users/anatolelevrel/Documents/M2 - ACO/Machine Learning/Projet Machine Learning")

csv_path = DIR/"dataset.csv"
source = DIR/"mes_images"
annotations = DIR/'folder_structure/yolov5/labels'
destination = DIR/"dataset_patient"

df = pd.read_csv(csv_path)

df["filestem"] = df["filestem"].astype(str)
df["patient_id"] = df["patient_id"].astype(str)

# fracture_visible vide = pas de fracture
df["fracture_visible"] = pd.to_numeric(
    df["fracture_visible"], errors="coerce"
)

# 2. Classer les patients

patients = (
    df.groupby("patient_id")["fracture_visible"]
    .apply(lambda x: x.notna().any())
    .rename("has_fracture")
    .reset_index()
)

patients_fracture = patients[
    patients["has_fracture"]
] # constitution du dataset avec les patients ayant une fracture

patients_sans_fracture = patients[
    ~patients["has_fracture"]
] # constitution du dataset avec les patients n'ayant pas de fracture

print("Patients avec fracture :", len(patients_fracture))
print("Patients sans fracture :", len(patients_sans_fracture))


# 3. Ratio 90 % / 10 %
# au niveau des patients :

# Garder tous les patients avec fracture
# et échantillonner les patients sans fracture.
nb_negatifs = round(len(patients_fracture) / 9)

if len(patients_sans_fracture) < nb_negatifs:
    raise ValueError("Pas assez de patients sans fracture.")

patients_negatifs_retenus = patients_sans_fracture.sample(
    n=nb_negatifs,
    random_state=42
) # sélection des patients sans fracture aléatoirement pour ratio 90/10

patients_retenus = pd.concat(
    [patients_fracture, patients_negatifs_retenus],
    ignore_index=True
).sample(frac=1, random_state=42) # ajout des patients avec fracture et mélange aléatoire

print("\nPatients sélectionnés :")
print(patients_retenus["has_fracture"].value_counts())


# 4. Split 70 / 15 / 15
# par patient

train_patients, temp_patients = train_test_split(
    patients_retenus,
    test_size=0.30,
    random_state=42,
    stratify=patients_retenus["has_fracture"]
) # séparation des patients retenus en train et temp (val + test) avec stratification (car peu de patients sans fracture)

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    random_state=42,
    stratify=temp_patients["has_fracture"]
) # séparation de temp_patients en val et test avec stratification (car peu de patients sans fracture)

splits = {
    "train": set(train_patients["patient_id"]),
    "val": set(val_patients["patient_id"]),
    "test": set(test_patients["patient_id"])
} # dictionnaire contenant les ensembles de patients pour chaque split


# 5. Associer les images
# aux patients sélectionnés

patient_map = dict(zip(df["filestem"], df["patient_id"])) # création d'un dictionnaire pour mapper les filestems aux patient_ids

images = []

for classe in ["fracture", "sans_fracture"]:
    dossier = source / classe # dossier avec images avec fracture et sans fracture

    for img in dossier.glob("*"):
        if not img.is_file(): 
            continue

        patient_id = patient_map.get(img.stem) # récupération du patient_id correspondant au filestem de l'image

        if patient_id is None:
            continue

        images.append({
            "path": img,
            "patient_id": patient_id,
            "classe": classe,
            "label": annotations / f"{img.stem}.txt" 
        }) # création d'une liste de dictionnaires contenant le chemin de l'image, le patient_id, la classe et le chemin du label correspondant

data = pd.DataFrame(images)


# 6. Copier les images
# et les annotations

if destination.exists():
    shutil.rmtree(destination) # suppression du dossier destination s'il existe déjà

for nom, patient_ids in splits.items():

    ensemble = data[
        data["patient_id"].isin(patient_ids)
    ]# sélection des images correspondant aux patients de l'ensemble actuel (train, val ou test)

    for _, ligne in ensemble.iterrows():

        dossier_images = destination / "images" / nom 
        dossier_labels = destination / "labels" / nom

        dossier_images.mkdir(parents=True, exist_ok=True)
        dossier_labels.mkdir(parents=True, exist_ok=True) # création des dossiers pour les images et les labels si ils n'existent pas

        shutil.copy2(
            ligne["path"],
            dossier_images / ligne["path"].name
        ) # copie de l'image dans le dossier correspondant à l'ensemble actuel (train, val ou test)

        label_dest = (
            dossier_labels / f"{ligne['path'].stem}.txt"
        ) # chemin de destination pour le label correspondant à l'image

        if ligne["classe"] == "fracture":
            if not ligne["label"].is_file():
                raise FileNotFoundError(
                    f"Annotation absente : {ligne['label']}"
                ) # vérification que le fichier d'annotation existe pour les images avec fracture
            lignes_fracture = []
            for line in ligne["label"].read_text(encoding="utf-8").splitlines(): 
                parts = line.split()
                if parts and int(parts[0]) == 3:      # récupération des annotations de classe 3 (fracture) uniquement
                    parts[0] = "0" # changement de l'ID de classe de 3 à 0 pour le dataset final
                    lignes_fracture.append(" ".join(parts))
            label_dest.write_text("\n".join(lignes_fracture) + ("\n" if lignes_fracture else ""), encoding="utf-8")
        else:
            label_dest.write_text("")

    print(f"\n{nom.upper()}")
    print("Patients :", ensemble["patient_id"].nunique())
    print("Images :", len(ensemble))
    print(ensemble["classe"].value_counts())


# 7. Vérification

assert splits["train"].isdisjoint(splits["val"]) # vérification que les ensembles train et val n'ont pas de patients en commun
assert splits["train"].isdisjoint(splits["test"])
assert splits["val"].isdisjoint(splits["test"])

print("\nDataset créé dans :", destination)
print("Aucun patient partagé entre les ensembles.")

Patients avec fracture : 3587
Patients sans fracture : 2504

Patients sélectionnés :
has_fracture
True     3587
False     399
Name: count, dtype: int64

TRAIN
Patients : 662
Images : 2651
classe
fracture         2301
sans_fracture     350
Name: count, dtype: int64

VAL
Patients : 144
Images : 572
classe
fracture         505
sans_fracture     67
Name: count, dtype: int64

TEST
Patients : 146
Images : 601
classe
fracture         528
sans_fracture     73
Name: count, dtype: int64

Dataset créé dans : /Users/anatolelevrel/Documents/M2 - ACO/Machine Learning/Projet Machine Learning/dataset_patient
Aucun patient partagé entre les ensembles.
